# Load data

In [66]:
import pandas as pd
import seaborn as sns
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

# Master
df_pduct = pd.read_parquet('../dataset/cleaned/products.parquet')
df_cus = pd.read_parquet('../dataset/cleaned/customers.parquet')
df_pmot = pd.read_parquet('../dataset/cleaned/promotions.parquet')
df_geo = pd.read_parquet('../dataset/cleaned/geography.parquet')

# Transaction
df_ord = pd.read_parquet('../dataset/cleaned/orders.parquet')
df_item = pd.read_parquet('../dataset/cleaned/order_items.parquet')
df_ship = pd.read_parquet('../dataset/cleaned/shipments.parquet')
df_ret = pd.read_parquet('../dataset/cleaned/returns.parquet')
df_rev = pd.read_parquet('../dataset/cleaned/reviews.parquet')

# Analytical
df_sale = pd.read_parquet('../dataset/cleaned/sales.parquet')
#df_submit = pd.read_parquet('../dataset/sample_submission.csv')

# Operational
df_inv = pd.read_parquet('../dataset/cleaned/inventory.parquet')
df_web = pd.read_parquet('../dataset/cleaned/web_traffic.parquet')

df_pduct.info()
df_cus.info()
df_pmot.info()
df_geo.info()
df_ord.info()
df_item.info()
df_ship.info()
df_ret.info()
df_rev.info()
df_sale.info()
df_inv.info()
df_web.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   product_id    2412 non-null   string  
 1   product_name  2412 non-null   string  
 2   category      2412 non-null   category
 3   segment       2412 non-null   category
 4   size          2412 non-null   category
 5   color         2412 non-null   category
 6   price         2412 non-null   float64 
 7   cogs          2412 non-null   float64 
dtypes: category(4), float64(2), string(2)
memory usage: 86.0 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   customer_id          121930 non-null  string        
 1   zip                  121930 non-null  string        
 2   city                 121930 non-null  cate

# gen file test

In [67]:
df_sale_train = df_sale[['Date', 'COGS']].copy()
df_sale_test = df_sale[['Date', 'COGS']].copy()

split_day = '2022-12-31'
df_sale_train = df_sale_train[(df_sale_train['Date'] <= split_day) & (df_sale_train['Date'] >= '2013-01-01')]
df_sale_test = df_sale_test[(df_sale_test['Date'] > split_day) & (df_sale_train['Date'] >= '2013-01-01')]

df_sale_train = df_sale_train.sort_values(by='Date')

In [68]:
df_sale_train['day'] = df_sale_train['Date'].dt.day
df_sale_train['month'] = df_sale_train['Date'].dt.month
df_sale_train['year'] = df_sale_train['Date'].dt.year
df_sale_train['month_day'] = df_sale_train['Date'].dt.strftime('%m-%d')

In [69]:
df_sale_train['is_Spring_promotion'] = df_sale_train['month_day'].between('03-18', '04-17')
df_sale_train['is_MidYear_promotion'] = df_sale_train['month_day'].between('06-23', '07-22')
df_sale_train['is_FallLauch_promotion'] = df_sale_train['is_Special_Fall_promotion'] = df_sale_train['month_day'].between('08-30', '10-01') | ((df_sale_train['year'] % 4 == 1) & (df_sale_train['month_day'] == '10-02'))
df_sale_train['is_YearEnd_promotion'] = df_sale_train['month_day'].between('11-18', '12-31') |  df_sale_train['month_day'].between('01-01', '01-02') # có nhiều ngày kết thúc khác nhưng fix vào 02/01 hàng năm
df_sale_train['is_Rural_promotion'] = (df_sale_train['year'] % 2 == 1)  & df_sale_train['month_day'].between('01-30', '03-01') # từ năm 2015 mới cố định ngày kết thúc
df_sale_train['is_Urban_promotion'] = (df_sale_train['year'] % 2 == 1)  & df_sale_train['month_day'].between('07-30', '09-02') 

In [70]:
df_sale_train['category_promotion'] = -1 # default: if it isn't in any promotion

#decode category_promotion: 0 -> All ; 1 -> Outdoor ; 2 -> Steetwear
df_sale_train.loc[(df_sale_train['is_Spring_promotion'] | df_sale_train['is_MidYear_promotion'] | df_sale_train['is_YearEnd_promotion']), 'category_promotion'] = 0
df_sale_train.loc[df_sale_train['is_Rural_promotion'], 'category_promotion'] = 1
df_sale_train.loc[df_sale_train['is_Urban_promotion'], 'category_promotion'] = 2

In [71]:
df_sale_train.to_parquet('sale_train_vi.parquet')

# Order status

In [72]:
df_check = df_ord[['order_id', 'order_date', 'order_status', 'payment_method']].copy()
df_check = df_check.merge(df_ship, on='order_id', how='left')

df_check['is_returned'] = df_check['order_id'].isin(df_ret['order_id']).astype(bool)
df_check['is_reviewed'] = df_check['order_id'].isin(df_rev['order_id']).astype(bool)

# df_check.info()
# print(df_check['order_status'].value_counts())

statuses = df_check['order_status'].unique()

for status in statuses:
    print("-" * 50)
    df_temp = df_check[df_check['order_status'] == status]
    total_rows = len(df_temp)
    
    count_return = len(df_temp[df_temp['is_returned'] == 1])
    count_ship = df_temp['ship_date'].notna().sum()
    
    print(f"--- order_status: {status.upper()} ---")
    print(f"Tổng số đơn: {total_rows}")
    print(f"Số đơn có tồn tại dữ liệu shipments: {count_ship}")
    print(f"Số đơn có tồn tại dữ liệu return: {count_return}")

--------------------------------------------------
--- order_status: DELIVERED ---
Tổng số đơn: 516716
Số đơn có tồn tại dữ liệu shipments: 516192
Số đơn có tồn tại dữ liệu return: 0
--------------------------------------------------
--- order_status: RETURNED ---
Tổng số đơn: 36142
Số đơn có tồn tại dữ liệu shipments: 36113
Số đơn có tồn tại dữ liệu return: 36062
--------------------------------------------------
--- order_status: SHIPPED ---
Tổng số đơn: 13773
Số đơn có tồn tại dữ liệu shipments: 13762
Số đơn có tồn tại dữ liệu return: 0
--------------------------------------------------
--- order_status: CANCELLED ---
Tổng số đơn: 59462
Số đơn có tồn tại dữ liệu shipments: 0
Số đơn có tồn tại dữ liệu return: 0
--------------------------------------------------
--- order_status: PAID ---
Tổng số đơn: 13577
Số đơn có tồn tại dữ liệu shipments: 0
Số đơn có tồn tại dữ liệu return: 0
--------------------------------------------------
--- order_status: CREATED ---
Tổng số đơn: 7275
Số đơn

# Inventory

In [73]:
df_inv.describe(include='all')
df_check = df_inv.sort_values(by='snapshot_date')
df_check['recei_sold'] = df_check['units_received'] - df_check['units_sold']
df_check = df_check[['snapshot_date', 'product_id', 'stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 'overstock_flag']]

df_temp = df_check[df_check['product_id'] == 1202]
display(df_temp.head(10))

df_temp = df_check[df_check['product_id'] == 1942]
display(df_temp.head(10))

,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,overstock_flag


,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,overstock_flag
